# Task E: Executable Transformer Orbital Baseline Notebook

这个 notebook 直接调用已经实现好的训练脚本：

- `train_transformer_orbital.py`
- `src/data_orbital.py`
- `src/models/transformer_forecaster.py`
- `src/train_utils.py`

推荐顺序：

1. 先构造一个小型 debug HDF5
2. 跑 debug 训练
3. 检查 `feature_metadata.json` 和 `horizon_metrics.csv`
4. 再跑正式参数

In [2]:
from pathlib import Path
import csv
import json
import subprocess
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
DATA_DIR = ROOT / "dataset_v3"
ICS_PATH = DATA_DIR / "ICs_sampling_2.hdf5"
RESULT_SHARD_PATH = DATA_DIR / "results_batch_00_sampling_2.hdf5"
DEBUG_DATA_PATH = Path("/tmp/orbital_smoke_sampling_2.hdf5")
DEBUG_OUTPUT_DIR = ROOT / "runs" / "debug_transformer_orbital"
FULL_OUTPUT_DIR = ROOT / "runs" / "transformer_orbital_baseline"
PYTHON_CMD = ["conda", "run", "--no-capture-output", "-n", "stellar", "python", "-u"]

assert ICS_PATH.exists(), f"Missing dataset file: {ICS_PATH}"
assert RESULT_SHARD_PATH.exists(), f"Missing result shard: {RESULT_SHARD_PATH}"

print(f"ROOT = {ROOT}")
print(f"ICS_PATH = {ICS_PATH}")
print(f"RESULT_SHARD_PATH = {RESULT_SHARD_PATH}")
print(f"DEBUG_DATA_PATH = {DEBUG_DATA_PATH}")
print(f"DEBUG_OUTPUT_DIR = {DEBUG_OUTPUT_DIR}")
print(f"FULL_OUTPUT_DIR = {FULL_OUTPUT_DIR}")

ROOT = /home/wan/stellar_dynamics
ICS_PATH = /home/wan/stellar_dynamics/dataset_v3/ICs_sampling_2.hdf5
RESULT_SHARD_PATH = /home/wan/stellar_dynamics/dataset_v3/results_batch_00_sampling_2.hdf5
DEBUG_DATA_PATH = /tmp/orbital_smoke_sampling_2.hdf5
DEBUG_OUTPUT_DIR = /home/wan/stellar_dynamics/runs/debug_transformer_orbital
FULL_OUTPUT_DIR = /home/wan/stellar_dynamics/runs/transformer_orbital_baseline


## Build A Small Debug Dataset

这个 cell 会从 `results_batch_00_sampling_2.hdf5` 取前 256 条轨迹，
并故意混用 `name` / `name_complete` 两种 key，顺带验证 loader 的健壮性。

In [3]:
import h5py

MIXED_KEY_MAPPING = {
    "aratios": "aratios_complete",
    "mratios_complete": "mratios_complete",
    "eccs": "eccs_complete",
    "incs_complete": "incs_complete",
    "APs": "APs_complete",
    "LANs_complete": "LANs_complete",
}
DEBUG_ROWS = 256

with h5py.File(RESULT_SHARD_PATH, "r") as src, h5py.File(DEBUG_DATA_PATH, "w") as dst:
    for out_key, in_key in MIXED_KEY_MAPPING.items():
        dst.create_dataset(out_key, data=src[in_key][:DEBUG_ROWS], compression="gzip")

print(f"Wrote debug dataset to {DEBUG_DATA_PATH}")
with h5py.File(DEBUG_DATA_PATH, "r") as f:
    for key in sorted(f.keys()):
        print(key, f[key].shape)

Wrote debug dataset to /tmp/orbital_smoke_sampling_2.hdf5
APs (256, 2001, 2)
LANs_complete (256, 2001, 2)
aratios (256, 2001, 1)
eccs (256, 2001, 2)
incs_complete (256, 2001, 2)
mratios_complete (256, 2001, 2)


## Helpers

这些函数用来运行训练脚本、检查输出文件、打印指标并展示图像。

In [4]:
REQUIRED_OUTPUTS = [
    "config.json",
    "normalization_stats.npz",
    "feature_metadata.json",
    "train_log.csv",
    "best_model.pt",
    "final_model.pt",
    "test_metrics.json",
    "horizon_metrics.csv",
    "loss_curve.png",
    "horizon_mse.png",
    "example_predictions.png",
]


def run_training(script_args):
    cmd = PYTHON_CMD + ["train_transformer_orbital.py"] + [str(arg) for arg in script_args]
    print("$", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)


def validate_outputs(output_dir: Path):
    missing = [name for name in REQUIRED_OUTPUTS if not (output_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing outputs in {output_dir}: {missing}")
    print(f"All expected outputs exist in {output_dir}")


def show_metrics(output_dir: Path, horizon_rows: int = 6):
    feature_metadata = json.load(open(output_dir / "feature_metadata.json", "r", encoding="utf-8"))
    test_metrics = json.load(open(output_dir / "test_metrics.json", "r", encoding="utf-8"))

    display(Markdown("### Feature Groups"))
    print(feature_metadata["selected_features"])
    for group in feature_metadata["groups"]:
        print(group["name"], (group["channel_start"], group["channel_end"]), group["channel_names"][:4])

    display(Markdown("### Test Metrics"))
    keys = [
        "mse",
        "mae",
        "aratios_mse",
        "mratios_mse",
        "eccs_mse",
        "incs_mse",
        "APs_mse",
        "LANs_mse",
        "best_epoch",
    ]
    print({key: test_metrics[key] for key in keys if key in test_metrics})

    display(Markdown("### Horizon Metrics Preview"))
    with open(output_dir / "horizon_metrics.csv", "r", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        rows = []
        for idx, row in enumerate(reader):
            rows.append(row)
            if idx + 1 >= horizon_rows:
                break
    for row in rows:
        print(row)


def show_plots(output_dir: Path):
    for name in ["loss_curve.png", "horizon_mse.png", "example_predictions.png"]:
        path = output_dir / name
        if path.exists():
            display(Markdown(f"### {name}"))
            display(Image(filename=str(path)))

## Run Debug Training

这里使用小样本 debug 数据，先确认整套产物链路都正常。

In [ ]:
run_training([
    "--data_path", DEBUG_DATA_PATH,
    "--context_len", 128,
    "--pred_len", 32,
    "--stride", 32,
    "--batch_size", 32,
    "--epochs", 3,
    "--d_model", 128,
    "--n_heads", 4,
    "--num_layers", 2,
    "--dim_feedforward", 512,
    "--dropout", 0.1,
    "--output_dir", DEBUG_OUTPUT_DIR,
    "--seed", 42,
    "--amp",
])

In [ ]:
validate_outputs(DEBUG_OUTPUT_DIR)
show_metrics(DEBUG_OUTPUT_DIR)
show_plots(DEBUG_OUTPUT_DIR)

## Run Full Training

这个 cell 使用真实数据入口 `dataset_v3/ICs_sampling_2.hdf5`。
脚本会自动解析到同目录下的 `results_batch_*_sampling_2.hdf5` 分片。

注意：这一步会明显更慢，也更占显存。

In [5]:
run_training([
    "--data_path", ICS_PATH,
    "--context_len", 256,
    "--pred_len", 64,
    "--stride", 32,
    "--batch_size", 64,
    "--epochs", 50,
    "--lr", 1e-4,
    "--weight_decay", 1e-4,
    "--d_model", 256,
    "--n_heads", 8,
    "--num_layers", 4,
    "--dim_feedforward", 1024,
    "--dropout", 0.1,
    "--output_dir", FULL_OUTPUT_DIR,
    "--seed", 42,
    "--amp",
])

$ conda run --no-capture-output -n stellar python -u train_transformer_orbital.py --data_path /home/wan/stellar_dynamics/dataset_v3/ICs_sampling_2.hdf5 --context_len 256 --pred_len 64 --stride 32 --batch_size 64 --epochs 50 --lr 0.0001 --weight_decay 0.0001 --d_model 256 --n_heads 8 --num_layers 4 --dim_feedforward 1024 --dropout 0.1 --output_dir /home/wan/stellar_dynamics/runs/transformer_orbital_baseline --seed 42 --amp
/home/wan/stellar_dynamics/src/data_orbital.py:383: RuntimeWarning: Detected NaN or inf values for feature 'aratios' in /home/wan/stellar_dynamics/dataset_v3/results_batch_00_sampling_2.hdf5; applying np.nan_to_num.
  transformed = sanitize_array(transformed, feature_name=group["name"], source_path=source.path)
/home/wan/stellar_dynamics/src/data_orbital.py:383: RuntimeWarning: Detected NaN or inf values for feature 'aratios' in /home/wan/stellar_dynamics/dataset_v3/results_batch_01_sampling_2.hdf5; applying np.nan_to_num.
  transformed = sanitize_array(transformed, f

KeyboardInterrupt: 

In [ ]:
validate_outputs(FULL_OUTPUT_DIR)
show_metrics(FULL_OUTPUT_DIR)
show_plots(FULL_OUTPUT_DIR)